In [6]:
import pandas as pd
import matplotlib.pyplot as plt

laps = pd.read_parquet("../data/processed/laps_model.parquet")
print(laps.shape)

(78233, 37)


In [7]:
import numpy as np

stint_keys = ["Year", "Round", "Driver", "Stint"]
MIN_STINT_LAPS = 8

laps["StintLaps"] = laps.groupby(stint_keys)["LapNumber"].transform("size")
long_stints = laps[laps["StintLaps"] >= MIN_STINT_LAPS].copy()

print("stints total:", laps.groupby(stint_keys).ngroups)
print("stints >= 8 laps:", long_stints.groupby(stint_keys).ngroups)

stints total: 4227
stints >= 8 laps: 3646


In [8]:
def stint_slope(group):
    slope, intercept = np.polyfit(group["TyreLife"], group["LapTimeSec"], deg=1)
    return slope

slopes = (
    long_stints
    .groupby(stint_keys + ["Compound"])[["TyreLife", "LapTimeSec"]]
    .apply(stint_slope)
    .reset_index(name="SecPerLap")
)
slopes.head()

,Year,Round,Driver,Stint,Compound,SecPerLap
0,2022,1,ALB,1.0,SOFT,0.233491
1,2022,1,ALB,2.0,MEDIUM,0.072089
2,2022,1,ALO,1.0,SOFT,0.240267
3,2022,1,ALO,2.0,MEDIUM,0.066098
4,2022,1,ALO,3.0,HARD,0.135510


In [9]:
summary = (
    slopes.groupby("Compound")["SecPerLap"]
    .agg(["median", "count"])
    .loc[["SOFT", "MEDIUM", "HARD"]]
)
summary

,median,count
Compound,,
SOFT,0.030189,586
MEDIUM,0.006299,1554
HARD,-0.005012,1506


In [10]:
long_stints = long_stints.sort_values(stint_keys + ["LapNumber"])

long_stints["LapInStint"] = long_stints.groupby(stint_keys).cumcount() + 1

long_stints["StintStart"] = (
    long_stints.groupby(stint_keys)["LapTimeSec"]
    .transform(lambda times: times.iloc[:3].median())
)

long_stints["DeltaSec"] = long_stints["LapTimeSec"] - long_stints["StintStart"]

In [11]:
curve = (
    long_stints.groupby(["Compound", "LapInStint"])["DeltaSec"]
    .agg(["median", "count"])
    .reset_index()
)

curve = curve[(curve["count"] >= 50) & (curve["LapInStint"] <= 35)]
curve.head()

,Compound,LapInStint,median,count
0,HARD,1,0.0135,1506
1,HARD,2,0.0000,1506
2,HARD,3,0.0000,1506
3,HARD,4,-0.0650,1506
4,HARD,5,-0.0720,1506


In [12]:
import plotly.express as px

tyre_colors = {"SOFT": "#E10600", "MEDIUM": "#FFD700", "HARD": "#9E9E9E"}

fig = px.line(
    curve,
    x="LapInStint",
    y="median",
    color="Compound",
    color_discrete_map=tyre_colors,
    category_orders={"Compound": ["SOFT", "MEDIUM", "HARD"]},
    markers=True,
    labels={
        "LapInStint": "Laps on this set of tyres",
        "median": "Slower than start of stint (s)",
        "Compound": "Tyre",
    },
    title="How lap times change as tyres age (2022-2025)",
)
fig.add_hline(y=0, line_dash="dash", line_color="grey")
fig.show()

In [14]:
fig.write_image("../images/degradation_curve.png", scale=2)

In [16]:
bins = [8, 15, 25, 100]
names = ["8-14 laps", "15-24 laps", "25+ laps"]

long_stints["StintLength"] = pd.cut(
    long_stints["StintLaps"], bins=bins, labels=names, right=False
)
long_stints.groupby("StintLength", observed=True)[stint_keys].nunique()

,Year,Round,Driver,Stint
StintLength,,,,
8-14 laps,4,24,31,6
15-24 laps,4,24,31,6
25+ laps,4,24,31,5


In [17]:
curve_len = (
    long_stints.groupby(["Compound", "StintLength", "LapInStint"], observed=True)["DeltaSec"]
    .agg(["median", "count"])
    .reset_index()
)
curve_len = curve_len[curve_len["count"] >= 20]

In [18]:
fig3 = px.line(
    curve_len,
    x="LapInStint",
    y="median",
    color="StintLength",
    facet_col="Compound",
    category_orders={"Compound": ["SOFT", "MEDIUM", "HARD"]},
    markers=True,
    labels={"LapInStint": "Laps on this set of tyres",
            "median": "Slower than start of stint (s)",
            "StintLength": "Stint length"},
    title="Tyre degradation by stint length",
)
fig3.add_hline(y=0, line_dash="dash", line_color="grey")
fig3.show()

In [19]:
fig3.write_image("../images/degradation_by_stint_length.png", scale=2, width=1400, height=500)